### Fetch UniParc IDs for RefSeq accessions that didn't map to a Uniprot ID

In [1]:
import re
import time
import json
import requests
from requests.adapters import HTTPAdapter, Retry
import pandas as pd


## This code was adapted from https://www.uniprot.org/help/id_mapping_prog


## Functions to retrieve UniParc ID
POLLING_INTERVAL = 3
API_URL = "https://rest.uniprot.org"

retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))


def check_response(response):
    try:
        response.raise_for_status()
    except requests.HTTPError:
        print(response.json())
        raise


def submit_id_mapping(from_db, to_db, ids):
    request = requests.post(
        f"{API_URL}/idmapping/run",
        data={"from": from_db, "to": to_db, "ids": ",".join(ids)},
    )
    check_response(request)
    return request.json()["jobId"]


def check_id_mapping_results_ready(job_id):
    while True:
        request = session.get(f"{API_URL}/idmapping/status/{job_id}")
        check_response(request)
        j = request.json()
        if "jobStatus" in j:
            if j["jobStatus"] in ("NEW", "RUNNING"):
                print(f"Retrying in {POLLING_INTERVAL}s")
                time.sleep(POLLING_INTERVAL)
            else:
                raise Exception(j["jobStatus"])
        else:
            return bool(j["results"] or j["failedIds"] or j["suggestedIds"])

def get_id_mapping_results_link(job_id):
    url = f"{API_URL}/idmapping/details/{job_id}"
    request = session.get(url)
    check_response(request)
    return request.json()["redirectURL"]


def get_id_mapping_results_search(url):
    results = requests.get(url)
    return results.json()

In [2]:
no_uniprot_acc=pd.read_excel("../data/defense_finder/refseq2uniprot_script/2025-06-24_RefSeq2Uniprot_mapping.xlsx",sheet_name="no_uniprot_matches")
no_uniprot_acc

,accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank
0,WP_001372321.1,Mok_Hok_Sok,Mok_Hok_Sok,"Klebsiella aerogenes,Escherichia coli,Citrobac...","GCF_022559325_NZ_CP060980_Mok_Hok_Sok_1,GCF_01...","GCF_002156845.1_NZ_CP021341_00030,GCF_00218019...","GCF_002156845.1_NZ_CP021341_00030,GCF_00218019...",692,5
1,WP_096937776.1,Mok_Hok_Sok,Mok_Hok_Sok,"Escherichia coli,Escherichia fergusonii,Klebsi...","GCF_016026215_NZ_CP065610_Mok_Hok_Sok_1,GCF_01...","GCF_010365485.1_NZ_CP048362_00038,GCF_02042404...","GCF_010365485.1_NZ_CP048362_00038,GCF_02042404...",262,25
2,WP_003917092.1,Cas,CAS_Class1-Subtype-III-A,"Mycobacterium tuberculosis,Mycobacterium orygis",GCF_014884645_NZ_CP043996_CAS_Class1-Subtype-I...,"GCF_002975475.1_NZ_CP027035_02939,GCF_01490083...","GCF_013010385.1_NZ_CP053092_02977,GCF_01691777...",257,27
3,WP_001381369.1,Cas,CAS_Class1-Subtype-I-E,"Shigella flexneri,Escherichia coli,Escherichia...",GCF_013167055_NZ_CP053605_CAS_Class1-Subtype-I...,"GCF_014075915.1_NZ_CP050031_00968,GCF_02260534...","GCF_014075795.1_NZ_CP050036_00976,GCF_00803329...",254,28
4,WP_020277900.1,MazEF,MazEF,"Klebsiella aerogenes,Enterobacter roggenkampii...","GCF_905330065_NZ_OW967005_MazEF_1,GCF_01677594...","GCF_011106605.1_NZ_CP040031_00014,GCF_02275994...","GCF_018274525.2_NZ_CP086665_00007,GCF_00190859...",215,39
...,...,...,...,...,...,...,...,...,...
122259,WP_086613943.1,BREX,BREX_I,Klebsiella pneumoniae,GCF_016925315_NZ_CP070438_BREX_I_2,GCF_016925315.1_NZ_CP070438_00016,GCF_016925315.1_NZ_CP070438_00021,1,202
122260,WP_086414403.1,RM,RM_Type_I,Eggerthella lenta,GCF_002148255_NZ_CP021140_RM_Type_I_1,GCF_002148255.1_NZ_CP021140_00899,GCF_002148255.1_NZ_CP021140_00901,1,202
122261,WP_086625893.1,RM,RM_Type_II,Enterobacter kobei,GCF_023333675_NZ_CP083862_RM_Type_II_2,GCF_023333675.1_NZ_CP083862_03290,GCF_023333675.1_NZ_CP083862_03291,1,202
122262,WP_086627698.1,Dpd,Dpd,Klebsiella pneumoniae,GCF_013305325_NZ_CP054303_Dpd_5,GCF_013305325.1_NZ_CP054303_05013,GCF_013305325.1_NZ_CP054303_05024,1,202


In [3]:
no_uniprot_acc_list=no_uniprot_acc["accession_in_sys"].tolist()
len(no_uniprot_acc_list),no_uniprot_acc_list

(122264,
 ['WP_001372321.1',
  'WP_096937776.1',
  'WP_003917092.1',
  'WP_001381369.1',
  'WP_020277900.1',
  'WP_000357807.1',
  'WP_159376494.1',
  'WP_001443175.1',
  'WP_223178619.1',
  'WP_000880885.1',
  'WP_000571783.1',
  'WP_000937120.1',
  'WP_000061856.1',
  'WP_000777275.1',
  'WP_010922512.1',
  'WP_001323520.1',
  'WP_001084115.1',
  'WP_167876495.1',
  'WP_000960581.1',
  'WP_071532149.1',
  'WP_168876894.1',
  'WP_006250003.1',
  'WP_000433154.1',
  'WP_004195968.1',
  'WP_032433805.1',
  'WP_032433488.1',
  'WP_000562556.1',
  'WP_009308509.1',
  'WP_004178703.1',
  'WP_004178705.1',
  'WP_019842341.1',
  'WP_004178700.1',
  'WP_000101606.1',
  'WP_000443455.1',
  'WP_003689382.1',
  'WP_223349259.1',
  'WP_001356249.1',
  'WP_001425460.1',
  'WP_223340727.1',
  'WP_001096907.1',
  'WP_002220782.1',
  'WP_000063179.1',
  'WP_032875608.1',
  'WP_000020778.1',
  'WP_012503916.1',
  'WP_003907189.1',
  'WP_001104341.1',
  'WP_012602456.1',
  'WP_000192872.1',
  'WP_00501

### Currently there is a limit of 100,000 IDs per API request so we're going to break this down

#### Submit job for batch 1

In [4]:
no_uniprot_acc_list_batch1=no_uniprot_acc_list[:100000]
len(no_uniprot_acc_list_batch1)

100000

In [5]:
print(no_uniprot_acc_list_batch1)

['WP_001372321.1', 'WP_096937776.1', 'WP_003917092.1', 'WP_001381369.1', 'WP_020277900.1', 'WP_000357807.1', 'WP_159376494.1', 'WP_001443175.1', 'WP_223178619.1', 'WP_000880885.1', 'WP_000571783.1', 'WP_000937120.1', 'WP_000061856.1', 'WP_000777275.1', 'WP_010922512.1', 'WP_001323520.1', 'WP_001084115.1', 'WP_167876495.1', 'WP_000960581.1', 'WP_071532149.1', 'WP_168876894.1', 'WP_006250003.1', 'WP_000433154.1', 'WP_004195968.1', 'WP_032433805.1', 'WP_032433488.1', 'WP_000562556.1', 'WP_009308509.1', 'WP_004178703.1', 'WP_004178705.1', 'WP_019842341.1', 'WP_004178700.1', 'WP_000101606.1', 'WP_000443455.1', 'WP_003689382.1', 'WP_223349259.1', 'WP_001356249.1', 'WP_001425460.1', 'WP_223340727.1', 'WP_001096907.1', 'WP_002220782.1', 'WP_000063179.1', 'WP_032875608.1', 'WP_000020778.1', 'WP_012503916.1', 'WP_003907189.1', 'WP_001104341.1', 'WP_012602456.1', 'WP_000192872.1', 'WP_005014373.1', 'WP_012503917.1', 'WP_001521154.1', 'WP_023283164.1', 'WP_022630855.1', 'WP_000063170.1', 'WP_08022

In [6]:
job_id = submit_id_mapping(
    from_db="RefSeq_Protein", to_db="UniProtKB", ids=no_uniprot_acc_list_batch1
)

In [7]:
print(job_id)

pkqEtDCTyz


#### Submit job for batch 2

In [13]:
no_uniprot_acc_list_batch2=no_uniprot_acc_list[100000:]
len(no_uniprot_acc_list_batch2)

22264

In [14]:
print(no_uniprot_acc_list_batch2)

['WP_120172199.1', 'WP_120173271.1', 'WP_119946308.1', 'WP_119944944.1', 'WP_119944956.1', 'WP_119945366.1', 'WP_119945367.1', 'WP_119945368.1', 'WP_119945425.1', 'WP_119945426.1', 'WP_119945427.1', 'WP_119945665.1', 'WP_119945666.1', 'WP_119945667.1', 'WP_119945833.1', 'WP_119945834.1', 'WP_119946031.1', 'WP_119946034.1', 'WP_119946254.1', 'WP_119946307.1', 'WP_119953980.1', 'WP_119985449.1', 'WP_119980130.1', 'WP_119980122.1', 'WP_114936010.1', 'WP_118663612.1', 'WP_115763930.1', 'WP_115765971.1', 'WP_115765978.1', 'WP_115766712.1', 'WP_115766781.1', 'WP_115766820.1', 'WP_115766867.1', 'WP_115766870.1', 'WP_115766872.1', 'WP_115766873.1', 'WP_115769682.1', 'WP_115770971.1', 'WP_115770972.1', 'WP_115776209.1', 'WP_115776210.1', 'WP_115777727.1', 'WP_115763895.1', 'WP_115780135.1', 'WP_115759545.1', 'WP_115759528.1', 'WP_115725099.1', 'WP_115725103.1', 'WP_115725104.1', 'WP_115725105.1', 'WP_115735561.1', 'WP_115736045.1', 'WP_115737005.1', 'WP_115747009.1', 'WP_115757426.1', 'WP_11575

In [15]:
job_id_2 = submit_id_mapping(
    from_db="RefSeq_Protein", to_db="UniProtKB", ids=no_uniprot_acc_list_batch2
)

print(job_id_2)

8IttPMUNt8


### Checking progress

#### Batch 1

In [23]:
if check_id_mapping_results_ready(job_id):
    link_batch1 = get_id_mapping_results_link(job_id)
    print(link_batch1)
    results_batch1 = get_id_mapping_results_search(link_batch1)
    print(results_batch1)

https://rest.uniprot.org/idmapping/uniprotkb/results/pkqEtDCTyz


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [24]:
batch1_failed_ids=results_batch1["failedIds"]

batch1_ids = pd.DataFrame(results_batch1["suggestedIds"])
batch1_ids

,from,to
0,WP_001372321.1,UPI00025C98FA
1,WP_096937776.1,UPI000BDF8CB7
2,WP_003917092.1,UPI0001E62B16
3,WP_001381369.1,UPI000013B026
4,WP_020277900.1,UPI00000B9DB1
...,...,...
92666,WP_120161578.1,UPI000BB9DD26
92667,WP_120161586.1,UPI000BB9E5F5
92668,WP_120161589.1,UPI000BB9D94F
92669,WP_120171879.1,UPI000E74A980


#### Batch 2

In [25]:
if check_id_mapping_results_ready(job_id_2):
    link_batch2 = get_id_mapping_results_link(job_id_2)
    print(link_batch2)
    results_batch2 = get_id_mapping_results_search(link_batch2)
    print(results_batch2)

https://rest.uniprot.org/idmapping/uniprotkb/results/8IttPMUNt8
{'results': [], 'failedIds': ['WP_080614269.1', 'WP_128485899.1', 'WP_072148972.1', 'WP_074163443.1', 'WP_126511658.1', 'WP_139909208.1', 'WP_100653316.1', 'WP_096745656.1', 'WP_116831561.1', 'WP_096720804.1', 'WP_100900584.1', 'WP_125219703.1', 'WP_099060725.1', 'WP_102991667.1', 'WP_115264245.1', 'WP_125277383.1', 'WP_075744804.1', 'WP_126527438.1', 'WP_107336033.1', 'WP_082815661.1', 'WP_129640702.1', 'WP_095842494.1', 'WP_104796129.1', 'WP_075093046.1', 'WP_137457086.1', 'WP_107336808.1', 'WP_123839739.1', 'WP_126372828.1', 'WP_083331111.1', 'WP_095858794.1', 'WP_086157595.1', 'WP_099061430.1', 'WP_096220535.1', 'WP_068393504.1', 'WP_075750697.1', 'WP_102992374.1', 'WP_128161570.1', 'WP_108209764.1', 'WP_079155248.1', 'WP_077194057.1', 'WP_071842897.1', 'WP_080528222.1', 'WP_080955907.1', 'WP_065819645.1', 'WP_086904612.1', 'WP_100654653.1', 'WP_073799795.1', 'WP_131265807.1', 'WP_096686641.1', 'WP_126483399.1', 'WP_07

In [26]:
batch2_failed_ids=results_batch2["failedIds"]

batch2_ids = pd.DataFrame(results_batch2["suggestedIds"])
batch2_ids

,from,to
0,WP_120172199.1,UPI000E74ADBD
1,WP_120173271.1,UPI000E711358
2,WP_119946308.1,UPI000E75E813
3,WP_119944944.1,UPI000E75BC82
4,WP_119944956.1,UPI000E726677
...,...,...
20833,WP_086613943.1,UPI000A39145F
20834,WP_086414403.1,UPI000A3BF26F
20835,WP_086625893.1,UPI000A3A7E04
20836,WP_086627698.1,UPI000A3D305F


### Concatenate dataframes and merge with no_uniprot_acc df

In [29]:
uniparc_ids=pd.concat([batch1_ids,batch2_ids])
uniparc_ids = uniparc_ids.rename(columns={'from': 'accession_in_sys', 'to': 'UniParc_ID'})
uniparc_ids

,accession_in_sys,UniParc_ID
0,WP_001372321.1,UPI00025C98FA
1,WP_096937776.1,UPI000BDF8CB7
2,WP_003917092.1,UPI0001E62B16
3,WP_001381369.1,UPI000013B026
4,WP_020277900.1,UPI00000B9DB1
...,...,...
20833,WP_086613943.1,UPI000A39145F
20834,WP_086414403.1,UPI000A3BF26F
20835,WP_086625893.1,UPI000A3A7E04
20836,WP_086627698.1,UPI000A3D305F


In [32]:
len(set(uniparc_ids["accession_in_sys"].tolist()))

113509

In [35]:
no_uniprot_acc_w_uniparc_ids=no_uniprot_acc.merge(uniparc_ids,on="accession_in_sys",how="left")
no_uniprot_acc_w_uniparc_ids

,accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,UniParc_ID
0,WP_001372321.1,Mok_Hok_Sok,Mok_Hok_Sok,"Klebsiella aerogenes,Escherichia coli,Citrobac...","GCF_022559325_NZ_CP060980_Mok_Hok_Sok_1,GCF_01...","GCF_002156845.1_NZ_CP021341_00030,GCF_00218019...","GCF_002156845.1_NZ_CP021341_00030,GCF_00218019...",692,5,UPI00025C98FA
1,WP_096937776.1,Mok_Hok_Sok,Mok_Hok_Sok,"Escherichia coli,Escherichia fergusonii,Klebsi...","GCF_016026215_NZ_CP065610_Mok_Hok_Sok_1,GCF_01...","GCF_010365485.1_NZ_CP048362_00038,GCF_02042404...","GCF_010365485.1_NZ_CP048362_00038,GCF_02042404...",262,25,UPI000BDF8CB7
2,WP_003917092.1,Cas,CAS_Class1-Subtype-III-A,"Mycobacterium tuberculosis,Mycobacterium orygis",GCF_014884645_NZ_CP043996_CAS_Class1-Subtype-I...,"GCF_002975475.1_NZ_CP027035_02939,GCF_01490083...","GCF_013010385.1_NZ_CP053092_02977,GCF_01691777...",257,27,UPI0001E62B16
3,WP_001381369.1,Cas,CAS_Class1-Subtype-I-E,"Shigella flexneri,Escherichia coli,Escherichia...",GCF_013167055_NZ_CP053605_CAS_Class1-Subtype-I...,"GCF_014075915.1_NZ_CP050031_00968,GCF_02260534...","GCF_014075795.1_NZ_CP050036_00976,GCF_00803329...",254,28,UPI000013B026
4,WP_020277900.1,MazEF,MazEF,"Klebsiella aerogenes,Enterobacter roggenkampii...","GCF_905330065_NZ_OW967005_MazEF_1,GCF_01677594...","GCF_011106605.1_NZ_CP040031_00014,GCF_02275994...","GCF_018274525.2_NZ_CP086665_00007,GCF_00190859...",215,39,UPI00000B9DB1
...,...,...,...,...,...,...,...,...,...,...
122259,WP_086613943.1,BREX,BREX_I,Klebsiella pneumoniae,GCF_016925315_NZ_CP070438_BREX_I_2,GCF_016925315.1_NZ_CP070438_00016,GCF_016925315.1_NZ_CP070438_00021,1,202,UPI000A39145F
122260,WP_086414403.1,RM,RM_Type_I,Eggerthella lenta,GCF_002148255_NZ_CP021140_RM_Type_I_1,GCF_002148255.1_NZ_CP021140_00899,GCF_002148255.1_NZ_CP021140_00901,1,202,UPI000A3BF26F
122261,WP_086625893.1,RM,RM_Type_II,Enterobacter kobei,GCF_023333675_NZ_CP083862_RM_Type_II_2,GCF_023333675.1_NZ_CP083862_03290,GCF_023333675.1_NZ_CP083862_03291,1,202,UPI000A3A7E04
122262,WP_086627698.1,Dpd,Dpd,Klebsiella pneumoniae,GCF_013305325_NZ_CP054303_Dpd_5,GCF_013305325.1_NZ_CP054303_05013,GCF_013305325.1_NZ_CP054303_05024,1,202,UPI000A3D305F


In [36]:
no_uniprot_acc_w_uniparc_ids.to_excel("../data/defense_finder/refseq2uniprot_script/2025-06-27_no_uniprot_acc_w_uniparc_ids.xlsx",index=False)